# Introduction to basic functionalities of flowerMD

## Overview:

This tutorial introduces some basic functionalities of the flowerMD package including:

- Initializing molecule structures
- Assembling molecules in a box
- Applying force fields
- Running MD simulations using different methods




## Step-by-step guide for running MD simulation for a box of  Polyphenylene sulfide (PPS) polymers:
In this tutorial, we will run a molecular dynamics simulation of Polyphenylene sulfide (PPS) polymers using the flowerMD package.
flowerMD uses the [`HOOMD-blue`](https://hoomd-blue.readthedocs.io/en/v4.1.0/) simulation engine in the background to run different methods of simulation, and utilizes many functionalities from [`mBuild`](https://mbuild.mosdef.org/en/stable/) and [GMSO](https://gmso.mosdef.org/en/stable/) to initialize molecular structures, apply forcefields and prepare the information necessary to run a simulation.

In summary, the flowerMD package has three main classes:

-  `Molecule`: This class is used to define the structure of a molecule (for example the structure of a polymer built from a monomer). This class allows us to implement recipies for building complex structures.

- `System`: This class is used to assemble molecules into a box and to capture the initial `gsd` snapshot of the system. It also applies the forcefiled to the system and prepares the required forces for the simulation.

- `Simulation`: This class is used to run the simulation using the `HOOMD-blue` simulation engine in the background. The Simulation class provides additional functionalities to run simulations following different processes/methods. 

In [1]:
import warnings

warnings.filterwarnings("ignore")

### Step 1: Initializing the Molecule
In this example, we are using the pre-defined recipe for building PPS molecules defined in flowerMD's library. The `PPS` class is a subclass of the `Molecule` class. This class includes all the necessary information for building the PPS chains, including the monomer structure and how the monomers bond to create a chain. All we need to specify is the polymer chain length and how many polymer chains we want to create. In this example, we will create a system of 10 PPS chains with a length of 8.

In [1]:
from flowermd.library import LJChain

molecules = LJChain(num_mols=10, lengths=8)
molecules.bond_lengths['A-A']
molecules.n_mols
molecules.lengths
for chain in molecules.molecules:
    print(chain.xyz)

[[0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 2.]
 [0. 0. 3.]
 [0. 0. 4.]
 [0. 0. 5.]
 [0. 0. 6.]
 [0. 0. 7.]]
[[0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 2.]
 [0. 0. 3.]
 [0. 0. 4.]
 [0. 0. 5.]
 [0. 0. 6.]
 [0. 0. 7.]]
[[0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 2.]
 [0. 0. 3.]
 [0. 0. 4.]
 [0. 0. 5.]
 [0. 0. 6.]
 [0. 0. 7.]]
[[0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 2.]
 [0. 0. 3.]
 [0. 0. 4.]
 [0. 0. 5.]
 [0. 0. 6.]
 [0. 0. 7.]]
[[0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 2.]
 [0. 0. 3.]
 [0. 0. 4.]
 [0. 0. 5.]
 [0. 0. 6.]
 [0. 0. 7.]]
[[0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 2.]
 [0. 0. 3.]
 [0. 0. 4.]
 [0. 0. 5.]
 [0. 0. 6.]
 [0. 0. 7.]]
[[0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 2.]
 [0. 0. 3.]
 [0. 0. 4.]
 [0. 0. 5.]
 [0. 0. 6.]
 [0. 0. 7.]]
[[0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 2.]
 [0. 0. 3.]
 [0. 0. 4.]
 [0. 0. 5.]
 [0. 0. 6.]
 [0. 0. 7.]]
[[0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 2.]
 [0. 0. 3.]
 [0. 0. 4.]
 [0. 0. 5.]
 [0. 0. 6.]
 [0. 0. 7.]]
[[0. 0. 0.]
 [0. 0. 1.]
 [0. 0. 2.]
 [0. 0. 3.]
 [0. 0. 4.]
 [0. 0. 5.]
 [0. 0. 6.]
 [0. 0. 7.]]


### Step 2: Initializing the System
In this step, we will use the `Pack` class, which is a subclass of the `System` class, to pack a box of PPS molecules in a random fashion at a given density (density unit is $g/cm^3$). The `System` class creates the box, organizes molecules within the box, applies the forcefield (if provided) to the system and creates
the initial state of the system in form of a HOOMD snapshot. 
The `apply_forcefield` method in `System`, applies the forcefield (works only for XML-based forcefields) to the system and generates the list of HOOMD force objects defining bonded and non-bonded interactions.

Alternatively, users can initiate their own custom list of HOOMD force objects. In such cases, there is no need to call `apply_forcefield` method or specify the `force_field` parameter during system setup. Instead, a list of `hoomd.md.force.Force` objects are passed directly to the `Simulation` class in the subsequent step. This approach allows for a greater flexibility in customizing the force interactions within the simulation, especially in cases where XML-based forcefields are not available. For more examples, see [the coarse graining tutorial](3-coarse-graining.ipynb).

#### Step 2.1: Creating the box and placing molecules




In this example, the `Pack` class invokes mBuild's `fill_box` method in the background, which efficiently places molecules within a box in a randomized manner without overlaps. This method uses [PACKMOL](https://m3g.github.io/packmol/) to fill the box.

**A note on `density` and `packing_expand_factor` parameters:**

Given a density in $\dfrac{g}{cm^3}$, the system class calculates a box length that corresponds to the specified density. We refer to calculated length as `target_box`. The `packing_expand_factor` multiplies this calulated box length by a factor and initializes the system based on the expanded box length. The reason for this expansion is that sometimes `PACKMOL` might fail to arrange molecules if the box size is too small. To address this, we suggest initially using an expand factor (default is 5) to initiate the system. Afterward, once the simulation object is created, we can shrink the simulation box to the desired target density.

The `density` parameter in the `Pack` class assumes $\dfrac{g}{cm^3}$ by default. It can work with other units, or even other kinds of density (e.g. number density instead of mass density). `flowerMD` uses the [unyt](https://github.com/yt-project/unyt). package for handling different kinds of units.

In [2]:
from flowermd.library import RandomWalk
import unyt as u

system = RandomWalk(molecules=molecules, density=0.5*u.Unit("nm**-3"))

[[[1.23416981 1.71962884 4.32876569]
  [1.84294363 0.93216883 4.2323231 ]
  [1.66674578 1.78616503 3.74278781]
  [2.29066254 1.05244973 4.01184123]
  [3.03811622 1.09811838 3.34909892]
  [3.84733569 1.57930988 3.01202536]
  [3.93257687 1.16862484 2.10424139]
  [3.96970945 0.9434645  3.07785527]]

 [[3.67127518 2.12326931 1.80679198]
  [2.91329204 1.56067694 2.13686982]
  [3.74526085 1.94002985 1.73200036]
  [4.07568654 2.62317696 1.08074858]
  [4.28792929 3.45073516 0.56103669]
  [3.3244002  3.56131759 0.31734995]
  [4.20980892 3.36083111 5.32683261]
  [4.82772214 2.68832592 0.3053302 ]]

 [[3.24811964 1.01374913 3.65228172]
  [4.10456249 0.623731   3.31406358]
  [3.43851376 0.02278696 2.87219782]
  [3.44139784 4.58773147 2.3685268 ]
  [4.00853536 3.98379651 2.92854251]
  [4.71599872 3.34687059 2.62224042]
  [4.53251149 2.68277481 3.34702262]
  [3.64683585 2.81076602 3.79333745]]

 [[5.11289258 1.34768508 5.15131943]
  [5.01968895 0.76440767 0.52939221]
  [5.41171545 1.54991768 0.05055

ValueError: setting an array element with a sequence.

In [15]:
import numpy as np
unitless_array = (np.array([1, 2, 1]) * u.Unit("g") / u.Unit("cm**3"))
unitless_array.to_value('g/cm**3')[0]

np.float64(1.0)

#### Step 2.2: Applying Forcefield

Now that the molecules are packed in the box, we can apply the forcefield and parameterize particle interactions.
We use the pre-defined `OPLS` forcefield class, which was created from the [OPLS](https://en.wikipedia.org/wiki/OPLS) XML forcefield, to parameterize particle interactions.

The flowerMD library offers some commonly used forcefields that can be employed to parameterize the interactions within specific systems. Please refer to [flowerMD's documentation](https://flowermd.readthedocs.io/en/latest/forcefields.html) for more examples.


We also specify the `r_cut` parameter, which is the cutoff distance for the non-bonded interactions. If `auto_scale` is set to `True`, all the parameters defined in forces will be scaled. For example, all the `epsilon` values of Leonard-Jones potentials are scaled based on the maximum `epsilon` value. Also, `scale_charges=True` will make the system charge neutral.

In [ ]:
from flowermd.library import Bead_Spring_DPD

system.apply_forcefield(
    r_cut=1.15, force_field=Bead_Spring_DPD())

The initial snapshot can be acquired from the `system.hoomd_snapshot` object.


In [ ]:
system.hoomd_snapshot

The list of HOOMD force objects applied to the system can also be obtained by accessing the `system.hoomd_forcefield` attribute. These forces correspond to the bonded and non-bonded interactions parameterized from the `OPLS` forcefield.

In [ ]:
hoomd_forces = system.hoomd_forcefield
hoomd_forces

Now, let's examine the parameters of the LJ pair force. As you can see, the values of `epsilon` have been scaled to fall within the range of  0 to 1. The scaling factor for the `epsilon` parameter, which is expressed in units of energy, can be retrieved  from `system.reference_energy`. 

In [ ]:
lj_force = hoomd_forces[3]

dict(lj_force.params)

### Step 3: Running the Simulation

Using the snapshot and force objects provided by the `System` class in the previous step, we can proceed to initialize the simulation. The `Simulation` class, a subclass of `hoomd.Simulation`, offers additional features and functionalities that automate simulation methods, such as updating box volume, welding process and tensile tests.

**A note about Logging**

In addition, the Simulation class provides the functionality to log snapshots of the simulation as a `gsd` trajectory file while the simulation is running. The frequency of saving these snapshots into the gsd file is controlled by the `gsd_write_freq` parameter. Furthermore, the path and name of the trajectory file is specified using the `gsd_file_name` parameter, with the default being "trajectory.gsd".

The simulation objects automatically log various simulation data, including timestep, potential energy, kinetic temperature, pressure, and volume. These data are saved in a text file, and you can specify the name of this file using the `log_file_name` parameter (the default is sim_data.txt). The frequency at which this data is logged can be set using the `log_write_freq` parameter. These features allow for the efficient monitoring and analysis of simulation progress and results.

In [ ]:
from flowermd.base import Simulation

sim = Simulation.from_system(
    system=system, gsd_write_freq=100, log_write_freq=100
)

Let's run the simulation for 1000 time steps using the NVT ensemble at a scaled temperature of 1.0. The default thermostat in flowerMD simulations is Nosé-Hoover thermostat. Users can modify the thermostat by specifying the `thermostat` parameter during the initialization of the simulation object. Users can select from a range of available thermostats, which are defined in  [`flowermd.utils.base_types.HOOMDThermostats`](https://github.com/cmelab/flowerMD/blob/main/flowermd/utils/base_types.py). For more detailed information about thermostats please refer to [HOOMD Blue documentation](https://hoomd-blue.readthedocs.io/en/stable/module-md-methods-thermostats.html).

In [ ]:
sim.dpd_sim_loop()
sim.save_restart_gsd()

In [ ]:
for writer in sim.operations.writers:
    if isinstance(writer, hoomd.write.GSD):
        writer.flush()

In [ ]:
sim_visualizer.frame = -1
sim_visualizer.view()

The simulation class also allows users to run the simulation under different conditions/ensembles such as NPT ensemble, NVE ensemble and Langevin dynamics. Check out [flowerMD's documentation](https://flowermd.readthedocs.io/en/latest/simulation.html) for more functionalities.


In the upcoming tutorials, we will explore a selection of features offered by the flowerMD package, highlighting how they can be customized to meet specific research requirements.